In [11]:
import pandas as pd
import os

os.makedirs('../data/analytics', exist_ok=True)

sales = pd.read_csv(
    '../data/curated/sales_curated.csv'
)

customers = pd.read_csv(
    '../data/processed/customers.csv'
)

In [12]:
# Calculate customer behavior

customer_behaviour = (
    sales.groupby('customer_id')
    .agg(
        total_orders=('so_id', 'nunique'),
        total_quantity=('quantity', 'sum'),
        total_revenue=('line_grand_total', 'sum')
    )
    .reset_index()
)

customer_behaviour.to_csv(
    '../data/analytics/customer_purchase_behaviour.csv',
    index=False
)


customer_behaviour.to_csv(
    '../data/analytics/customer_purchase_behaviour.csv',
    index=False
)

In [13]:
# RFM Analysis

sales['order_date'] = pd.to_datetime(
    sales['order_date']
)

snapshot_date = (
    sales['order_date'].max()
    + pd.Timedelta(days=1)
)


rfm = (
    sales.groupby('customer_id')
    .agg({
        'order_date':
        lambda x:
        (snapshot_date - x.max()).days,

        'so_id': 'nunique',

        'line_grand_total': 'sum'
    })
)

rfm.columns = [
    'Recency',
    'Frequency',
    'Monetary'
]

rfm = rfm.reset_index()

rfm.to_csv(
    '../data/analytics/customer_rfm.csv',
    index=False
)

In [14]:
# customer segmentation using RFM analysis


rfm['segment'] = 'Bronze'

rfm.loc[
    rfm['Monetary']
    > rfm['Monetary'].quantile(.50),
    'segment'
] = 'Silver'

rfm.loc[
    rfm['Monetary']
    > rfm['Monetary'].quantile(.80),
    'segment'
] = 'Gold'

rfm.to_csv(
    '../data/analytics/customer_segments.csv',
    index=False
)

In [15]:
#Customer Lifetime Value (CLV) Analysis

customers['clv'] = (
    customers['total_purchase_value']
    /
    customers['customer_tenure_years']
)

customers[
    ['customer_id', 'clv']
].to_csv(
    '../data/analytics/customer_clv.csv',
    index=False
)

In [16]:
#customer Retention Analysis

customers['retained'] = (
    customers['days_since_last_purchase']
    <= 90
).astype(int)

customers[
    ['customer_id', 'retained']
].to_csv(
    '../data/analytics/customer_retention.csv',
    index=False
)

In [17]:
#Customer Churn Analysis

customers['churn_risk'] = (
    customers['days_since_last_purchase']
    > 180
).astype(int)

customers[
    ['customer_id',
     'churn_risk']
].to_csv(
    '../data/analytics/customer_churn.csv',
    index=False
)

In [18]:
# High Value Customer Analysis

high_value = (
    customers.sort_values(
        'total_purchase_value',
        ascending=False
    )
    .head(20)
)

high_value.to_csv(
    '../data/analytics/high_value_customers.csv',
    index=False
)

In [19]:
# Customer Credit risk

customers['credit_risk'] = 'Low'

customers.loc[
    customers[
        'credit_utilization_pct'
    ] > 0.80,
    'credit_risk'
] = 'High'

customers[
    ['customer_id',
     'credit_risk']
].to_csv(
    '../data/analytics/customer_credit_risk.csv',
    index=False
)

In [20]:
print("""
Customer Analytics Completed!

Generated:

1. customer_purchase_behaviour.csv
2. customer_rfm.csv
3. customer_segments.csv
4. customer_clv.csv
5. customer_retention.csv
6. customer_churn.csv
7. high_value_customers.csv
8. customer_credit_risk.csv
""")


Customer Analytics Completed!

Generated:

1. customer_purchase_behaviour.csv
2. customer_rfm.csv
3. customer_segments.csv
4. customer_clv.csv
5. customer_retention.csv
6. customer_churn.csv
7. high_value_customers.csv
8. customer_credit_risk.csv

